# Sesión 06 - Lab Reto 2: CRM y clickstream a escala real (sin solución)

Los labs anteriores de esta sesión trabajaron con datasets chicos, pensados para que cada técnica se vea con claridad en pocas filas. Este segundo reto va al extremo opuesto: un CRM de clientes y un log de sesiones de navegación a la escala con la que realmente se nota el efecto de `shuffle.partitions` y `autoBroadcastJoinThreshold`, algo que con pocas filas es imposible de percibir.

El escenario: la cadena de electrodomésticos de las Sesiones 04-05 tiene, además de su ERP, un CRM propio con cientos de miles de clientes y un registro de sesiones de navegación en su tienda online, cada una con un array anidado de eventos (páginas vistas, clics, agregados al carrito, compras). Los dos datasets llegan con los mismos problemas de calidad que ya viste en las Sesiones 05 y 06 (nulos, mayúsculas/minúsculas inconsistentes, reingestas exactas y versionadas), pero a un volumen que obliga a pensar en particiones, no solo en sintaxis: `clientes_crm_reto2` termina con cerca de 670.000 filas, y `eventos_clickstream_reto2` arranca en cerca de 233.000 sesiones que, al aplanar los eventos anidados, se convierten en cerca de 800.000 filas.

Vas a aplicar, sobre datos de esa escala: aplanamiento de estructuras anidadas, estandarización de texto, separación de una columna compuesta, filtros complejos, deduplicación exacta y versionada, agregaciones de negocio, enriquecimiento con `broadcast()`, y una medición real de tuning, no simulada.

## Generación de los datos de origen (setup, no es parte del reto)

A diferencia de los demás notebooks de la sesión, acá los archivos no vienen ya preparados en el volume: se generan en las siguientes celdas, con Spark, a la escala real que vas a analizar. Corré estas celdas una sola vez antes de empezar el reto. No hace falta entenderlas en detalle, no son parte del ejercicio.

Nota: al escribir a este volumen, Spark reparte cada archivo en varias partes (`part-00000...`, `part-00001...`, etc.) dentro de una carpeta, en vez de un único archivo: es el patrón real de escritura distribuida que vas a encontrar trabajando con datasets grandes. Tanto `spark.read.csv()` como `spark.read.json()` leen la carpeta completa sin que tengas que hacer nada distinto.

La generación puede tardar algunos minutos según el compute disponible.

In [0]:
from pyspark.sql import functions as F

RUTA_RETO2 = "/Volumes/dbassociate/default/vol_landing/sesion_06/reto2"

# Mas particiones durante la generacion para paralelizar mejor -- se resetea al valor
# de la sesion (200) antes de empezar el reto
spark.conf.set("spark.sql.shuffle.partitions", 64)

### Perfiles de clientes (CRM)

In [0]:
# genera información aleatoria en base al número que le pasemos
N_CLIENTES = 600000

ciudades_provincias = [
    "Lima - Lima", "Arequipa - Arequipa", "Cusco - Cusco", "Trujillo - La Libertad",
    "Chiclayo - Lambayeque", "Piura - Piura", "Huancayo - Junin", "Ica - Ica",
    "Tacna - Tacna", "Iquitos - Loreto",
]
estados_cuenta = ["activo", "inactivo", "suspendido"]
segmentos = ["Bronze", "Plata", "Oro", "Diamante"]
nombres = [
    "Maria", "Jose", "Ana", "Luis", "Carmen", "Carlos", "Rosa", "Miguel", "Patricia", "Jorge",
    "Lucia", "Fernando", "Diana", "Ricardo", "Gabriela", "Manuel", "Sofia", "Victor", "Elena", "Raul",
]
apellidos = [
    "Garcia", "Rodriguez", "Gonzalez", "Flores", "Mendoza", "Vasquez", "Castillo", "Torres",
    "Ramos", "Chavez", "Rojas", "Quispe", "Huaman", "Salazar", "Cardenas", "Paredes", "Aguilar",
    "Vega", "Reyes", "Campos",
]

def lista_a_columna(valores, seed):
    arreglo = F.array(*[F.lit(v) for v in valores])
    indice = (F.rand(seed=seed) * len(valores)).cast("int") + 1
    return F.element_at(arreglo, indice)

clientes_base = (
    spark.range(0, N_CLIENTES)
    .withColumnRenamed("id", "seq")
    .withColumn("cliente_id", F.format_string("CLI-%06d", F.col("seq")))
    .withColumn("_nombre_raw", lista_a_columna(nombres, 1))
    .withColumn("_apellido_raw", lista_a_columna(apellidos, 2))
    # Mayusculas/minusculas inconsistentes a proposito, mismo problema de clientes_crm.csv
    # de la Sesion 05, ahora a escala real
    .withColumn(
        "nombre",
        F.when(F.rand(seed=3) < 0.34, F.upper("_nombre_raw"))
         .when(F.rand(seed=4) < 0.5, F.lower("_nombre_raw"))
         .otherwise(F.initcap("_nombre_raw")),
    )
    .withColumn(
        "apellido",
        F.when(F.rand(seed=5) < 0.34, F.upper("_apellido_raw"))
         .when(F.rand(seed=6) < 0.5, F.lower("_apellido_raw"))
         .otherwise(F.initcap("_apellido_raw")),
    )
    .withColumn(
        "_email_base",
        F.concat_ws(".", F.lower("_nombre_raw"), F.lower("_apellido_raw"), F.col("seq").cast("string")),
    )
    # Emails con espacios sueltos y mayusculas al azar, para forzar trim()/lower() en el reto
    .withColumn(
        "email",
        F.when(F.rand(seed=7) < 0.15, F.concat(F.lit("  "), F.col("_email_base"), F.lit("@correo.com  ")))
         .when(F.rand(seed=8) < 0.5, F.upper(F.concat(F.col("_email_base"), F.lit("@correo.com"))))
         .otherwise(F.concat(F.col("_email_base"), F.lit("@correo.com"))),
    )
    .withColumn("email", F.when(F.rand(seed=9) < 0.03, F.lit(None)).otherwise(F.col("email")))
    .withColumn("ciudad_provincia", lista_a_columna(ciudades_provincias, 10))
    .withColumn(
        "ciudad_provincia",
        F.when(F.rand(seed=11) < 0.2, F.concat(F.lit("  "), F.col("ciudad_provincia"), F.lit("  ")))
         .otherwise(F.col("ciudad_provincia")),
    )
    .withColumn(
        "telefono",
        F.when(
            F.rand(seed=12) < 0.85,
            F.concat(F.lit("9"), (F.rand(seed=13) * 100000000).cast("long").cast("string")),
        ).otherwise(F.lit(None)),
    )
    .withColumn(
        "segmento",
        F.when(F.rand(seed=14) < 0.92, lista_a_columna(segmentos, 15)).otherwise(F.lit(None)),
    )
    .withColumn("estado_cuenta", lista_a_columna(estados_cuenta, 16))
    .withColumn("fecha_registro", F.date_sub(F.current_date(), (F.rand(seed=17) * 1500).cast("int")))
    .withColumn("ultima_actualizacion", F.current_timestamp())
    # ~0.3% sin cliente_id -- llave de negocio rota, no solo un campo opcional vacio
    .withColumn("cliente_id", F.when(F.rand(seed=18) < 0.003, F.lit(None)).otherwise(F.col("cliente_id")))
    .select(
        "cliente_id", "nombre", "apellido", "email", "ciudad_provincia", "telefono",
        "segmento", "estado_cuenta", "fecha_registro", "ultima_actualizacion",
    )
)

# Reingesta identica (byte a byte) de una muestra -- dropDuplicates() sin subset debe resolverla
duplicados_exactos = clientes_base.sample(fraction=0.08, seed=21)

# Reingesta con una version actualizada del mismo cliente_id -- exige row_number()
candidatos_version = clientes_base.filter(F.col("cliente_id").isNotNull()).sample(fraction=0.04, seed=22)
duplicados_version = (
    candidatos_version
    .withColumn("estado_cuenta", F.lit("activo"))
    .withColumn("segmento", lista_a_columna(segmentos, 23))
    .withColumn("ultima_actualizacion", F.expr("current_timestamp() + INTERVAL 1 HOUR"))
)

clientes_crm = clientes_base.unionByName(duplicados_exactos).unionByName(duplicados_version)

(
    clientes_crm
    .coalesce(16)
    .write.mode("overwrite")
    .option("header", True)
    .csv(f"{RUTA_RETO2}/clientes_crm_reto2")
)

### Sesiones de navegación con eventos anidados

In [0]:
N_SESIONES = 220000

tipos_evento = ["pageview", "click", "add_to_cart", "purchase", "search"]
paginas = [
    "/home", "/producto/101", "/producto/205", "/producto/318", "/carrito",
    "/checkout", "/busqueda", "/cuenta", "/ofertas",
]
dispositivos = ["Mobile", "Desktop", "Tablet"]

sesiones_base = (
    spark.range(0, N_SESIONES)
    .withColumnRenamed("id", "seq")
    .withColumn("sesion_id", F.format_string("SES-%06d", F.col("seq")))
    # 20% de sesiones son de visitantes anonimos, sin cliente_id
    .withColumn(
        "cliente_id",
        F.when(F.rand(seed=30) < 0.2, F.lit(None)).otherwise(
            F.format_string("CLI-%06d", (F.rand(seed=31) * N_CLIENTES).cast("int"))
        ),
    )
    .withColumn("_disp_raw", lista_a_columna(dispositivos, 32))
    .withColumn("dispositivo", F.when(F.rand(seed=33) < 0.5, F.upper("_disp_raw")).otherwise(F.col("_disp_raw")))
    .withColumn("fecha_sesion", F.date_sub(F.current_date(), (F.rand(seed=34) * 90).cast("int")))
    # Entre 1 y 6 eventos por sesion
    .withColumn("num_eventos", (F.rand(seed=35) * 6).cast("int") + 1)
    .select("sesion_id", "cliente_id", "dispositivo", "fecha_sesion", "num_eventos")
)

eventos_expandidos = (
    sesiones_base
    .withColumn("pos", F.explode(F.sequence(F.lit(0), F.col("num_eventos") - 1)))
    .withColumn("tipo_evento", lista_a_columna(tipos_evento, 36))
    .withColumn("pagina", lista_a_columna(paginas, 37))
    .withColumn(
        "timestamp_evento",
        (F.current_timestamp().cast("long") - (F.col("num_eventos") - F.col("pos")) * 60).cast("timestamp"),
    )
)

sesiones_anidadas = (
    eventos_expandidos
    .groupBy("sesion_id", "cliente_id", "dispositivo", "fecha_sesion")
    .agg(F.collect_list(F.struct("tipo_evento", "pagina", "timestamp_evento")).alias("eventos"))
)

# Reingesta identica de una muestra de sesiones completas (mismo problema que en el CSV)
sesiones_duplicadas = sesiones_anidadas.sample(fraction=0.06, seed=38)
sesiones_finales = sesiones_anidadas.unionByName(sesiones_duplicadas)

(
    sesiones_finales
    .coalesce(24)
    .write.mode("overwrite")
    .json(f"{RUTA_RETO2}/eventos_clickstream_reto2")
)

### Tabla de referencia de segmentos

In [0]:
segmentos_ref = spark.createDataFrame(
    [
        ("Bronze", "Envio estandar", 0),
        ("Plata", "Envio prioritario", 5),
        ("Oro", "Envio prioritario + soporte dedicado", 10),
        ("Diamante", "Envio prioritario + soporte dedicado + acceso anticipado", 15),
    ],
    ["segmento", "beneficio_principal", "descuento_pct"],
)

segmentos_ref.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{RUTA_RETO2}/segmentos_clientes_reto2"
)

# A partir de aca, el reto arranca desde el valor default de la sesion de Spark
spark.conf.set("spark.sql.shuffle.partitions", 200)

## Verificación del entorno

In [0]:
dbutils.fs.ls(RUTA_RETO2)

## Parte 1: Leer ambas fuentes con esquema explícito

Definí un `StructType` explícito para cada fuente. A esta escala, dejar que Spark infiera el schema (`inferSchema`) recorre el dataset completo dos veces y es notablemente más lento que en los labs con archivos chicos. Para `eventos_clickstream_reto2`, el array `eventos` necesita un `ArrayType(StructType([...]))`, mismo patrón que `mensajes`/`respuestas` en los labs anteriores de esta sesión.

In [0]:
# TODO: define el StructType de clientes_crm_reto2 (cliente_id, nombre, apellido, email,
#   ciudad_provincia, telefono, segmento, estado_cuenta, fecha_registro, ultima_actualizacion)
# schema_clientes = StructType([...])

# TODO: define el StructType de eventos_clickstream_reto2, incluyendo un array de eventos:
#   sesion_id (string), cliente_id (string, nullable), dispositivo (string), fecha_sesion (date),
#   eventos (array de struct con tipo_evento, pagina, timestamp_evento)
# schema_eventos = StructType([...])

# TODO: lee ambas carpetas con su schema explicito (recorda que cada una es una carpeta con
#   varios archivos part-, no un unico archivo)
# df_clientes_raw = spark.read.schema(schema_clientes).option("header", True).csv(f"{RUTA_RETO2}/clientes_crm_reto2")
# df_eventos_raw = spark.read.schema(schema_eventos).json(f"{RUTA_RETO2}/eventos_clickstream_reto2")

## Parte 2: Explorar antes de aplanar

Antes de tocar nada, medí el tamaño real de lo que estás por procesar: ¿cuántas filas trae cada fuente? ¿Cuántas sesiones son de visitantes anónimos (`cliente_id` nulo)? ¿Cuántos eventos trae cada sesión en promedio, con `size()`? ¿Hay filas de `clientes_crm_reto2` sin `cliente_id` (una llave de negocio rota, no solo un campo opcional vacío)?

In [0]:
# TODO: cuenta filas de df_clientes_raw y df_eventos_raw

# TODO: cuenta cuantas sesiones tienen cliente_id nulo (visitantes anonimos)

# TODO: usa size() sobre la columna eventos para ver la distribucion de eventos por sesion
#   (min, max, promedio -- pista: agg con min()/max()/avg() sobre size("eventos"))

# TODO: cuenta cuantas filas de clientes tienen cliente_id nulo -- son filas que no se pueden
#   usar como llave de negocio, decidi mas adelante si las descartas

## Parte 3: Aplanar los eventos

`eventos` cambia el grano: de "una sesión" a "un evento de una sesión". Elegí `explode()` o `posexplode()` según si necesitás saber cuál fue el primer evento de cada sesión (la página de entrada, útil para medir por dónde entran los usuarios a la tienda).

In [0]:
# TODO: aplana "eventos" con la funcion que corresponda segun si necesitas la posicion
# df_eventos_planos = ...

# TODO: a partir de df_eventos_planos, queda con la primera pagina vista de cada sesion
#   (posicion 0) -- es la pagina de entrada (landing page)
# df_landing_pages = ...

## Parte 4: Estandarización y columnas compuestas

`nombre`, `apellido` y `email` llegan con mayúsculas/minúsculas inconsistentes y, en el caso de `email`, con espacios sueltos. `ciudad_provincia` combina dos datos en un solo campo, separados por `" - "` (con espacios extra alrededor en algunas filas). Dejá ambas columnas listas para usarse como llave de agrupación o de join.

In [0]:
# TODO: estandariza nombre y apellido con initcap(), y email con trim() + lower()

# TODO: separa ciudad_provincia en dos columnas (ciudad, provincia) con split()
#   -- ojo con los espacios extra alrededor del separador, aplica trim() antes o despues de separar

# TODO: dropea las columnas auxiliares/originales que ya no necesites y renombra si hace falta

## Parte 5: Filtros complejos

Sobre el DataFrame de clientes ya estandarizado, quedate solo con clientes `activo` de segmento `Oro` o `Diamante` que además tengan `telefono` registrado (para una campaña de retención por llamada). Combiná las tres condiciones en un solo filtro.

In [0]:
# TODO: filtra clientes con estado_cuenta == "activo", segmento in ("Oro", "Diamante") y
#   telefono no nulo, combinando las condiciones con &
# df_campana_retencion = ...

## Parte 6: Deduplicación a escala

`clientes_crm_reto2` trae dos problemas distintos, mezclados: una porción de filas reingresadas de forma idéntica (mismo criterio que `dropDuplicates()` sin `subset`), y otra porción de clientes reingresados con una versión más reciente (mismo `cliente_id`, `ultima_actualizacion` más nueva y otros campos cambiados, necesita `row_number()` sobre una ventana). `eventos_clickstream_reto2` trae solo el primer tipo: sesiones completas reingresadas de forma idéntica.

Pensá el orden: ¿conviene deduplicar las sesiones antes o después de aplanar los eventos? ¿Por qué?

In [0]:
# TODO: elimina las sesiones reingresadas de forma identica con dropDuplicates() sin subset
#   -- decidi si lo haces antes o despues de explode(), y por que

# TODO: sobre los clientes, elimina primero las reingestas identicas con dropDuplicates()

# TODO: despues, usa row_number() sobre una ventana particionada por cliente_id, ordenada por
#   ultima_actualizacion descendente, para quedarte con la version mas reciente de cada
#   cliente_id que fue reingresado con datos actualizados
# df_clientes_deduplicados = ...

# Pista: dropDuplicates() sin subset nunca va a detectar los duplicados versionados -- son filas
#   distintas fila por fila, solo comparten cliente_id

## Parte 7: Aterrizar en Bronze y Silver

Con los datos ya deduplicados (pero antes de las transformaciones de negocio), escribí las tablas Bronze con las columnas de auditoría de siempre. Después, con las transformaciones de las Partes 3-4 ya aplicadas, escribí las tablas Silver: `dbassociate.silver.clientes_reto2` y `dbassociate.silver.eventos_detalle_reto2`.

In [0]:
# TODO: agrega columnas de auditoria (ingestion_timestamp, source_system, batch_id) y escribe
#   dbassociate.bronze.clientes_crm_reto2 y dbassociate.bronze.eventos_clickstream_reto2

# TODO: escribe las versiones ya limpias/aplanadas/deduplicadas como
#   dbassociate.silver.clientes_reto2 y dbassociate.silver.eventos_detalle_reto2

## Parte 8: Agregaciones de negocio

Con las tablas Silver ya escritas, calculá: cantidad de eventos por tipo (`count`), clientes únicos que generaron al menos un evento (`approx_count_distinct` sobre `cliente_id`, sin contar visitantes anónimos), promedio de eventos por sesión (`mean`), y un `summary()` de esa misma métrica.

In [0]:
# TODO: cuenta eventos por tipo_evento (count)

# TODO: calcula approx_count_distinct(cliente_id) sobre los eventos con cliente_id no nulo

# TODO: calcula el promedio de eventos por sesion (mean)

# TODO: usa summary() sobre esa misma metrica (eventos por sesion, ya agrupada)

## Parte 9: Enriquecimiento con broadcast()

Uní el DataFrame de clientes deduplicados con `segmentos_clientes_reto2` (4 filas, candidato obvio a `broadcast()`) para calcular el descuento (`descuento_pct`) que le corresponde a cada cliente según su segmento. Sumale la cantidad de eventos generados por cada segmento y escribí el resultado como `dbassociate.silver.metricas_eventos_segmento_reto2`.

In [0]:
# TODO: lee segmentos_clientes_reto2 y unelo con broadcast() a los clientes por segmento

# TODO: agrega la cantidad de eventos generados por cada segmento (join con los eventos ya
#   aplanados y deduplicados, por cliente_id)

# TODO: escribe el resultado en dbassociate.silver.metricas_eventos_segmento_reto2

## Parte 10: Medí el tuning con datos reales

A esta escala, el efecto de `shuffle.partitions` y `autoBroadcastJoinThreshold` sí se nota en el reloj, no solo en el plan de ejecución. Medí con `time.time()` un `groupBy` sobre los eventos (por ejemplo, conteo por `tipo_evento`) con `shuffle.partitions` en su valor actual y de nuevo con un valor bien más chico (por ejemplo, 8). Después, repetí el join de la Parte 9 con `autoBroadcastJoinThreshold` en su valor default y en `-1`, comparando tanto el plan (`.explain()`) como el tiempo real.

In [0]:
import time

# TODO: guarda el valor actual de spark.sql.shuffle.partitions, medi el tiempo de un groupBy
#   sobre los eventos (por ejemplo, conteo por tipo_evento) con ese valor, y confirma en el
#   plan (.explain()) el numero de particiones del shuffle (nodo Exchange, hashpartitioning(..., N))

# TODO: ajusta spark.sql.shuffle.partitions a un valor bien mas chico (por ejemplo, 8) y repeti
#   la misma medicion de tiempo y de plan

# TODO: con autoBroadcastJoinThreshold en su valor default, corre .explain() sobre el join de
#   la Parte 9 y confirma que el plan elige BroadcastHashJoin

# TODO: baja autoBroadcastJoinThreshold a -1, repeti el .explain() y confirma que el plan cambia
#   a SortMergeJoin -- compara tambien el tiempo real de ambas versiones

# No te olvides de volver shuffle.partitions y autoBroadcastJoinThreshold a sus valores
# originales al terminar

## Preguntas de reflexión (sin respuesta única)

- ¿Por qué a esta escala el ajuste de `shuffle.partitions` sí se nota en el tiempo real, cuando en los labs guiados de la sesión (con datasets de pocas filas) el cambio no era medible?
- ¿Por qué `dropDuplicates()` sin `subset` nunca hubiera resuelto los duplicados versionados de `clientes_crm_reto2`, sin importar cuántas veces lo aplicaras?
- Algunos `cliente_id` de `eventos_clickstream_reto2` no existen en `clientes_crm_reto2` (podés confirmarlo con un `LEFT ANTI JOIN`). ¿De dónde podría venir esa inconsistencia en un sistema real, y en qué capa (Bronze, Silver o Gold) tendría sentido detectarla?
- Si en vez de `coalesce(16)` la generación de `clientes_crm_reto2` no hubiera limitado el número de archivos de salida, ¿qué problema práctico esperarías al leer esa carpeta más adelante?

## Limpieza

In [0]:
# TODO: elimina las tablas que hayas creado en este notebook, por ejemplo:
# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.clientes_crm_reto2")
# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.eventos_clickstream_reto2")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.clientes_reto2")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.eventos_detalle_reto2")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.metricas_eventos_segmento_reto2")

# Opcional: los archivos generados en el volumen ocupan varios cientos de MB -- si no los vas
# a reusar, podes borrarlos para liberar espacio
# dbutils.fs.rm(RUTA_RETO2, recurse=True)